In [10]:
from scipy import stats
from itertools import combinations
from statsmodels.stats.multitest import multipletests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
df = pd.read_csv('Taekwondo_Technique_Classification_Stats.csv', header=None, index_col=0)
technique = df.iloc[0]
participant = df.iloc[1]
trial = df.iloc[2]
data = df.iloc[4:].copy()
column_names = [technique, participant, trial]
data.columns = pd.MultiIndex.from_arrays(column_names, names=['Technique', 'Participant', 'Trial'])
data = data.T
data = data.dropna(how='all')
data = data.apply(pd.to_numeric, errors='coerce')
max_values = data.max(axis=1)
power = max_values - 1026
power_df = power.reset_index(name='Power')
power_df.to_csv('power_df.csv', index=False)

In [12]:
grouped = power_df.groupby(['Participant', 'Technique'])
mean_power = grouped['Power'].mean()
mean_power_df = mean_power.reset_index(name='Mean_Power')
mean_power_df.to_csv('mean_power_df.csv', index=False)

In [13]:
corr_df = mean_power_df.copy()
var_power = power_df.groupby(['Participant', 'Technique'])['Power'].var()
var_power = var_power.reset_index(name='Variance')
corr_df = pd.merge(corr_df, var_power, on=['Participant', 'Technique'], how='left')
athletes_df = pd.read_csv('Table1.csv')
athletes_df = athletes_df.rename(columns={'Participant ID': 'Participant', 'Weight (kg)': 'Weight', 'Experience (years)': 'Experience'})
athletes_df = athletes_df[['Participant', 'Age', 'Weight', 'Experience']]
corr_df = pd.merge(corr_df, athletes_df, on='Participant', how='left')
techniques = corr_df['Technique'].unique()

for tech in techniques:
    rows = corr_df['Technique'] == tech
    subset = corr_df[rows]
    corr_df.loc[rows, 'Corr_Power_Weight'] = subset['Mean_Power'].corr(subset['Weight'], method='spearman')
    corr_df.loc[rows, 'Corr_Power_Age'] = subset['Mean_Power'].corr(subset['Age'], method='spearman')
    corr_df.loc[rows, 'Corr_Power_Experience'] = subset['Mean_Power'].corr(subset['Experience'], method='spearman')

corr_df.to_csv('corr_df.csv', index=False)
corr_summary = corr_df.groupby('Technique')[['Corr_Power_Weight', 'Corr_Power_Age', 'Corr_Power_Experience']].first()
corr_summary = corr_summary.reset_index()
corr_summary.to_csv('corr_summary.csv', index=False)

In [14]:
features = ['Mean_Power', 'Weight', 'Age', 'Experience']
techniques = sorted(corr_df['Technique'].unique())

for tech in techniques:
    subset = corr_df[corr_df['Technique'] == tech]
    subset = subset[features]
    corr_matrix = subset.corr(method='spearman')
    plt.figure(figsize=(6, 5))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True, cbar_kws={'label': 'Spearman correlation'})
    plt.title('Correlation matrix — Technique ' + str(tech))
    plt.tight_layout()
    plt.savefig('corr_matrix_' + str(tech) + '.png', dpi=150)
    plt.close()

overall_subset = corr_df[features]
overall_matrix = overall_subset.corr(method='spearman')
plt.figure(figsize=(6, 5))
sns.heatmap(overall_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True, cbar_kws={'label': 'Spearman correlation'})
plt.title('Correlation matrix — All techniques')
plt.tight_layout()
plt.savefig('corr_matrix_overall.png', dpi=150)
plt.close()

In [15]:
friedman_var = corr_df.pivot(index='Participant', columns='Technique', values='Variance')
friedman_var = friedman_var[['R', 'B', 'C', 'P']]
friedman_var = friedman_var.dropna()
print(friedman_var)
groups_r = friedman_var['R'].values
groups_b = friedman_var['B'].values
groups_c = friedman_var['C'].values
groups_p = friedman_var['P'].values
stat_var, p_var = stats.friedmanchisquare(groups_r, groups_b, groups_c, groups_p)
friedman_var_df = pd.DataFrame({'Test': ['Friedman_Variance'], 'P_Value': [p_var]})
friedman_var_df.to_csv('friedman_variance.csv', index=False)

Technique          R        B      C       P
Participant                                 
P1             320.3   7954.2  119.5    78.0
P2            4294.7   1948.5   17.3  1290.3
P3            2133.8    435.7  156.7  1273.7
P4            2911.3   1832.2  399.2  2039.3
P5           11814.5  19163.7  730.0    88.2


In [16]:
power_df['Trial'] = pd.to_numeric(power_df['Trial'], errors='coerce')
trial_results = []
for tech in sorted(power_df['Technique'].unique()):
    subset = power_df[power_df['Technique'] == tech]
    trial_pivot = subset.pivot(index='Participant', columns='Trial', values='Power')
    trial_pivot = trial_pivot[[1, 2, 3, 4, 5]]
    trial_pivot = trial_pivot.dropna()
    t1 = trial_pivot[1].values
    t2 = trial_pivot[2].values
    t3 = trial_pivot[3].values
    t4 = trial_pivot[4].values
    t5 = trial_pivot[5].values
    stat, p = stats.friedmanchisquare(t1, t2, t3, t4, t5)
    trial_results.append({'Technique': tech, 'Statistic': stat, 'P_Value': p})
trial_results_df = pd.DataFrame(trial_results)
trial_results_df.to_csv('friedman_trial.csv', index=False)

In [17]:
mean_pivot = corr_df.pivot(index='Participant', columns='Technique', values='Mean_Power')
mean_pivot = mean_pivot[['R', 'B', 'C', 'P']]
technique_corr = mean_pivot.corr(method='spearman')
technique_corr.to_csv('technique_corr.csv')
plt.figure(figsize=(6, 5))
sns.heatmap(technique_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True, cbar_kws={'label': 'Spearman correlation'})
plt.title('Correlation between techniques')
plt.tight_layout()
plt.savefig('technique_corr.png', dpi=150)
plt.close()